# AutoStrat quick test

Run setup once, edit the prompt, then run **Generate**. Only the accepted DSL appears; diagnostics are collapsed below. No microscope commands are executed.

Use the EvoMachine `.venv` notebook kernel with the matching AutoStrat checkout. The existing `explore_autostrat_pipeline.ipynb` remains available for deeper virtual-hardware tests.

Schema 9: use matching `collection-loops-state` checkouts of EvoMachine and AutoStrat. Restart the kernel after switching branches. Variables use `name = expression`; scoped observations use `observations.NAME`; `loop fovs:` / nested `loop rois:` iterate registered collections without implicit hardware actions.

Microscopy pack 0.7.0 adds `image(detect_rois=true, segment=true, ...)` for DeLTA. Clear projection targets, call `select_roi()` for qualifying ROIs, then call `project_selected(...)` once after the ROI loop to expose them together. Use explicit waits and a requested stopping condition. Generation here never loads models or operates hardware; restart the kernel after updating the pack.


In [ ]:
import asyncio
import os
from getpass import getpass
from html import escape
from pathlib import Path

from IPython.display import Code, HTML, display
from autostrat import StrategyPipeline, load_domain_pack
from autostrat.generation import GeneratorConfig, PromptRecipe
from autostrat.verification import SemanticVerifierConfig
from evomachine.strategy_generation.preview import generate_preview

root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "evomachine/domain_packs/microscopy/domain.yaml").is_file()
)
domain = load_domain_pack(root / "evomachine/domain_packs/microscopy")

# Same defaults as the existing test notebook; environment variables override them.
os.environ.setdefault("OPENAI_BASE_URL", "https://robin-office-2.tail32bb7.ts.net/v1")
model_id = os.getenv("AUTOSTRAT_MODEL_ID", "qwen3.6")
model = model_id if ":" in model_id else f"openai-chat:{model_id}"
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Model API key: ")

pipeline = StrategyPipeline(
    domain,
    generator_config=GeneratorConfig(model=model, validation_retries=2),
    verifier_config=SemanticVerifierConfig(model=model, output_retries=2),
    prompt_recipe=PromptRecipe(name="quick-test", few_shot_count=3),
    semantic_revisions=2,
)
print(f"Ready: {model} | microscopy {domain.metadata.version}")


## Prompt

In [ ]:
prompt = """
During initialisation, move to the first field of view.
At each step, terminate if 6 steps have completed.
Otherwise wait for (step_count + 1) / 2 seconds, bounded between 0.5 and 3 seconds.
During finalisation, move to the first field of view.
"""


## Generate

The accepted DSL is shown below. Model calls use the endpoint configured above.

In [ ]:
# Clear the previous result before starting another request.
preview = None
preview = await asyncio.to_thread(generate_preview, pipeline, prompt)

if preview.verified is not None:
    display(HTML("<h3>DSL</h3>"))
    display(Code(preview.dsl, language="text"))
else:
    display(HTML("<b>No accepted strategy.</b><pre>" + escape(str(preview.error)) + "</pre>"))

display(HTML(
    "<details><summary>Diagnostics — attempts, revisions and errors</summary><pre>"
    + escape(preview.diagnostics())
    + "</pre></details>"
))


### Optional inspection

`preview.verified` holds the accepted result; `preview.attempts` holds semantic candidates and verdicts. For the full generation prompt, inspect `preview.verified.accepted.generated.prompt.messages` after success.

Successful-run deterministic retry counts are not currently exposed by AutoStrat and are reported as unavailable, not zero. This notebook checks generation and validation, not live observation values or hardware behaviour. Use the original notebook for virtual execution.